# Multipole-to-multipole translation (M2M)

## Purpose

M2M moves a child-centred multipole expansion to a parent centre. In a full
FMM this combines child expansions during the upward pass. Here we isolate one
child-like source region and compare its translated coefficients with P2M
built directly at the parent centre.

## Mathematical definition

With $d=c_{\mathrm{parent}}-c_{\mathrm{child}}$, the Cartesian shift is

$$M^{\mathrm{parent}}_\alpha =
\sum_{\gamma\le\alpha}\frac{d^\gamma}{\gamma!}
M^{\mathrm{child}}_{\alpha-\gamma}.$$

All coefficients through order $p$ are retained. We also compare the field
from each expansion against direct P2P at distant targets.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 4
n_sources = 40
random_seed = 42
child_centre = np.array([0.25, 0.25, 0.25])
parent_centre = np.array([0.0, 0.0, 0.0])
child_half_width = 0.20
target_distance = 4.0

## Problem setup

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = child_centre + rng.uniform(
    -child_half_width,
    child_half_width,
    size=(n_sources, 3),
)
dipole_moments = rng.normal(size=(n_sources, 3))

child_multipole = cdfmm.p2m_dipole(
    child_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
translated_parent = cdfmm.m2m(
    child_multipole,
    child_centre,
    parent_centre,
    order=expansion_order,
)
direct_parent = cdfmm.p2m_dipole(
    parent_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)

coefficient_difference = translated_parent - direct_parent
print(f"Coefficient count: {len(translated_parent)}")
print(f"Maximum coefficient difference: {np.max(np.abs(coefficient_difference)):.3e}")
print(f"Relative coefficient-vector difference: {np.linalg.norm(coefficient_difference) / np.linalg.norm(direct_parent):.3e}")

## Geometry

The purple arrow runs from the child expansion centre to the parent expansion centre. The source particles remain fixed.

In [ ]:
figure, axes = new_3d_figure()
axes.scatter(*source_positions.T, s=20, color="tab:blue", label="sources")
axes.scatter(*child_centre, marker="o", s=90, color="tab:orange", label="child centre")
axes.scatter(*parent_centre, marker="D", s=110, color="tab:red", label="parent centre")
draw_box_3d(axes, child_centre, child_half_width, colour="tab:orange", label="child box")
draw_box_3d(axes, parent_centre, 0.5, colour="tab:red", label="parent box")
axes.quiver(
    child_centre[0],
    child_centre[1],
    child_centre[2],
    *(parent_centre - child_centre),
    color="tab:purple",
    linewidth=2.0,
    arrow_length_ratio=0.18,
    label="M2M translation",
)
finish_3d_axes(axes, "M2M moves the source representation, not the particles")
axes.legend()
figure.tight_layout()

## Far-field comparison and convergence

In [ ]:
target_directions = random_unit_vectors(rng, 12)
target_positions = target_distance * target_directions
reference_fields = direct_fields(target_positions, source_positions, dipole_moments)

orders = np.arange(1, 7)
child_errors = []
translated_errors = []
direct_parent_errors = []

for order in orders:
    child_M = cdfmm.p2m_dipole(
        child_centre,
        source_positions,
        dipole_moments,
        order=order,
    )
    translated_M = cdfmm.m2m(
        child_M,
        child_centre,
        parent_centre,
        order=order,
    )
    parent_M = cdfmm.p2m_dipole(
        parent_centre,
        source_positions,
        dipole_moments,
        order=order,
    )

    child_fields = multipole_fields(target_positions, child_M, child_centre, order)
    translated_fields = multipole_fields(target_positions, translated_M, parent_centre, order)
    parent_fields = multipole_fields(target_positions, parent_M, parent_centre, order)
    child_errors.append(error_metrics(child_fields, reference_fields)["rms"])
    translated_errors.append(error_metrics(translated_fields, reference_fields)["rms"])
    direct_parent_errors.append(error_metrics(parent_fields, reference_fields)["rms"])

figure, axes = plt.subplots(figsize=(8, 5))
axes.semilogy(orders, child_errors, "o-", label="P2M(child) + M2P")
axes.semilogy(orders, translated_errors, "s-", label="P2M + M2M(parent) + M2P")
axes.semilogy(orders, direct_parent_errors, "^-", label="P2M(parent) + M2P")
axes.set_xlabel("Expansion order p")
axes.set_ylabel("RMS relative field error")
axes.set_title("M2M far-field convergence")
axes.legend()
figure.tight_layout()

## What to observe

For this single child, M2M reproduces the directly constructed parent
coefficients to floating-point precision. The child- and parent-centred
expansions have different convergence geometry, while the M2M route closely
tracks direct P2M at the parent. In a full upward pass, several child vectors
would be added to one parent vector.